# Assignment 5b: Pandas Groupby with Hurricane Data

**At-home assignment — worth 10 points.** When you're done, push your work to your week folder and post a link to your completed **notebook** on the matching Courseworks assignment.

In this assignment you'll apply `groupby` and time-series operations to the NOAA IBTrACS hurricane dataset.

:::{admonition} Learning goals
:class: tip
This assignment exercises the groupby and time-series skills from this section:

- Read a large CSV file with custom column-type handling
- Rename columns and inspect unique values
- Filter and `nlargest` operations on a `DataFrame`
- Use `groupby` to count, aggregate, and iterate over groups
- Plot scatter plots, bar charts, and hexbin plots from a DataFrame
- Set a datetime index and use it for time-based filtering
- Compute climatologies and anomalies with `groupby` + `transform`
:::

:::{admonition} Working through this notebook
:class: tip
**Download** this notebook using the ⬇ button in the top-right (or copy-paste the cells into a fresh notebook), open it in your environment (JupyterLab on LEAP or Colab), and fill in your solution under each numbered task.

When you're done, follow the [submission instructions](#submission-instructions) at the bottom of the page.
:::

Start by importing numpy, pandas, and matplotlib's pyplot. The cell below is empty — type your imports there and run it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

Use the following code to load a CSV file of the [NOAA IBTrACS](https://www.ncdc.noaa.gov/ibtracs/index.php?name=ibtracs-data) hurricane dataset:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

url = 'https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r00/access/csv/ibtracs.ALL.list.v04r00.csv'
df = pd.read_csv(url, parse_dates=['ISO_TIME'], usecols=range(12),
                 skiprows=[1], na_values=[' ', 'NOT_NAMED'],
                 keep_default_na=False, dtype={'NAME': str})
df.head()

Basin Key: (NI - North Indian, SI - South Indian, WP - Western Pacific, SP - Southern Pacific, EP - Eastern Pacific, NA - North Atlantic)

How many rows does this dataset have?

In [ ]:
len(df)

How many North Atlantic hurricanes are in this dataset?

In [ ]:
df[df["BASIN"] == "NA"]["SID"].nunique()

## 1) Get the unique values of the `BASIN`, `SUBBASIN`, and `NATURE` columns

In [ ]:
df["BASIN"].unique()

In [ ]:
df["SUBBASIN"].unique()

In [ ]:
df["NATURE"].unique()

## 2) Rename the `WMO_WIND` and `WMO_PRES` columns to `WIND` and `PRES`

In [ ]:
df = df.rename(columns={"WMO_WIND": "WIND", "WMO_PRES": "PRES"})
for col in ["WIND", "PRES", "LAT", "LON"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df.head()

## 3) Get the 10 largest rows in the dataset by `WIND`

In [ ]:
df.nlargest(10, "WIND")

You will notice some names are repeated.

## 4) Group the data on `SID` and get the 10 largest hurricanes by `WIND`

In [ ]:
df.groupby("SID")["WIND"].max().nlargest(10)

## 5) Make a bar chart of the wind speed of the 20 strongest-wind hurricanes

Use the name on the x-axis.

In [ ]:
strongest = (df.groupby("SID")
              .agg(WIND=("WIND", "max"), NAME=("NAME", "first"))
              .nlargest(20, "WIND"))
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(strongest["NAME"], strongest["WIND"], color="steelblue")
ax.set_ylabel("Max wind (knots)"); ax.set_title("20 strongest-wind hurricanes")
plt.xticks(rotation=90); plt.tight_layout()

## 6) Plot the count of all datapoints by Basin

as a bar chart

In [ ]:
df["BASIN"].value_counts().plot.bar(figsize=(10, 5), title="Datapoints per basin")

## 7) Plot the count of unique hurricanes by Basin

as a bar chart.

In [ ]:
df.groupby("BASIN")["SID"].nunique().plot.bar(figsize=(10, 5), title="Unique hurricanes per basin")

## 8) Make a `hexbin` of the location of datapoints in Latitude and Longitude

In [ ]:
df.plot.hexbin(x="LON", y="LAT", gridsize=40, figsize=(12, 6), cmap="viridis")

## 9) Find Hurricane Katrina (from 2005) and plot its track as a scatter plot

First find the SID of this hurricane.

In [ ]:
katrina_sid = df[(df["NAME"] == "KATRINA") & (df["SEASON"] == 2005)]["SID"].unique()[0]
katrina_sid

Next get this hurricane's group and plot its position as a scatter plot. Use wind speed to color the points.

In [ ]:
katrina = df[df["SID"] == katrina_sid]
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(katrina["LON"], katrina["LAT"], c=katrina["WIND"], cmap="plasma", s=40)
fig.colorbar(sc, ax=ax, label="wind (knots)")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude"); ax.set_title("Hurricane Katrina (2005) track")

## 10) Make time the index on your dataframe

In [ ]:
df = df.set_index("ISO_TIME")
df.head()

## 11) Plot the count of all datapoints per year as a timeseries

You should use `resample`

In [ ]:
df.resample("YE").size().plot(marker="o", figsize=(12, 5), title="Datapoints per year")

## 12) Plot all tracks from the North Atlantic in 2005

You will probably have to iterate through a `GroupBy` object

In [ ]:
na_2005 = df[(df["BASIN"] == "NA") & (df.index.year == 2005)]
fig, ax = plt.subplots(figsize=(12, 7))
for sid, g in na_2005.groupby("SID"):
    ax.plot(g["LON"], g["LAT"], marker=".")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude"); ax.set_title("North Atlantic tracks, 2005")

## 13) Create a filtered dataframe that contains only data since 1970 from the North Atlantic ("NA") Basin

Use this for the rest of the assignment

In [ ]:
df_na = df[(df["BASIN"] == "NA") & (df.index.year >= 1970)]
print(len(df_na), "rows")
df_na.head()

## 14) Plot the number of datapoints per day from this filtered dataframe

Make sure you figure is big enough to actually see the plot

In [ ]:
df_na.resample("D").size().plot(figsize=(16, 4), title="North Atlantic datapoints per day (since 1970)")

## 15) Calculate the climatology of datapoint counts as a function of `dayofyear`

Plot the mean and standard deviation on a single figure

In [ ]:
daily = df_na.resample("D").size()
clim = daily.groupby(daily.index.dayofyear).agg(["mean", "std"])
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(clim.index, clim["mean"], label="mean")
ax.fill_between(clim.index, clim["mean"] - clim["std"], clim["mean"] + clim["std"],
                alpha=0.3, label="± 1 std")
ax.set_xlabel("day of year"); ax.set_ylabel("datapoint count"); ax.legend()
ax.set_title("Climatology of daily NA datapoint counts")

## 16) Use `transform` to calculate the anomaly of daily counts from the climatology

Resample the anomaly timeseries at annual resolution and plot a line with dots as markers.

In [ ]:
anomaly = daily.groupby(daily.index.dayofyear).transform(lambda x: x - x.mean())
anomaly.resample("YE").mean().plot(marker="o", figsize=(12, 5),
                                   title="Annual-mean anomaly of daily NA counts")

Which years stand out as having anomalous hurricane activity?

## Submission instructions

When you're done, save your completed notebook as `assignment5b.ipynb` inside the current week's folder in your private `clmt5405-assignments` GitHub repo. Then push the commit:

```bash
git add <weekN>/assignment5b.ipynb
git commit -m "Submit assignment 5b"
git push
```

Due Sunday night.